In [9]:
import sqlite3
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")

COL_INSEE_INCENDIES = "Code INSEE"
COL_INSEE_COMMUNES = "code_insee"
COL_DATE = "Date de première alerte"
COL_SURFACE = "Surface parcourue (m2)"


In [10]:
def read_incendies(path, sep=";"):
    lines = [l for l in path.read_text(encoding="utf-8", errors="replace").splitlines() if l.strip()]
    counts = [len(l.split(sep)) for l in lines[:40]]
    n_cols = max(set(counts), key=counts.count)
    header_row = next(i for i, c in enumerate(counts) if c == n_cols)
    return pd.read_csv(path, sep=sep, skiprows=header_row)


files = sorted(DATA_DIR.glob("Incendies_*.csv"))
incendies = pd.concat([read_incendies(f) for f in files], ignore_index=True)

print(incendies.shape)
incendies.head()


(140248, 24)


,Année,Numéro,Département,Code INSEE,Nom de la commune,Date de première alerte,Surface parcourue (m2),Surface forêt (m2),Surface maquis garrigues (m2),Autres surfaces naturelles hors forêt (m2),...,Surfaces non boisées artificialisées (m2),Surfaces non boisées (m2),Précision des surfaces,Type de peuplement,Nature,Décès ou bâtiments touchés,Nombre de décès,Nombre de bâtiments totalement détruits,Nombre de bâtiments partiellement détruits,Précision de la donnée
0,1973,8,06,06060,Falicon,1973-01-09 13:50:00,10000,4700.0,4800.0,0.0,...,NaN,500.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN
1,1973,1871,83,83038,Châteaudouble,1973-01-28 13:25:00,30000,14100.0,14400.0,0.0,...,NaN,1500.0,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN
2,1973,1872,83,83078,Moissac-Bellevue,1973-01-29 11:15:00,6000,2820.0,2880.0,0.0,...,NaN,300.0,NaN,1.0,Involontaire (travaux),NaN,NaN,NaN,NaN,NaN
3,1973,1639,34,34163,Montarnaud,1973-01-29 13:00:00,100000,47000.0,48000.0,0.0,...,NaN,5000.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN
4,1973,9,06,06012,Beausoleil,1973-01-29 15:00:00,40000,18800.0,19200.0,0.0,...,NaN,2000.0,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN


## Chargement des communes

In [11]:
communes = pd.read_csv(DATA_DIR / "communes-france-2024.csv")

print(communes.shape)
communes.head()


(34990, 39)


/tmp/ipykernel_56261/3986801999.py:1: DtypeWarning: Columns (0: code_insee, 1: dep_code, 2: canton_code, 3: epci_code, 4: code_insee_centre_zone_emploi) have mixed types. Specify dtype option on import or set low_memory=False.
  communes = pd.read_csv(DATA_DIR / "communes-france-2024.csv")


,Unnamed: 0,code_insee,nom_standard,nom_sans_pronom,nom_a,nom_de,nom_sans_accent,nom_standard_majuscule,typecom,typecom_texte,...,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie,latitude_centre,longitude_centre,grille_densite,gentile,url_wikipedia,url_villedereve
0,0,01001,L'Abergement-Clémenciat,Abergement-Clémenciat,à Abergement-Clémenciat,de l'Abergement-Clémenciat,l-abergement-clemenciat,L'ABERGEMENT-CLÉMENCIAT,COM,commune,...,206.0,272.0,46.153,4.926,46.153,4.926,Rural à habitat dispersé,NaN,https://fr.wikipedia.org/wiki/fr:L'Abergement-...,https://villedereve.fr/ville/01001-l-abergemen...
1,1,01002,L'Abergement-de-Varey,Abergement-de-Varey,à Abergement-de-Varey,de l'Abergement-de-Varey,l-abergement-de-varey,L'ABERGEMENT-DE-VAREY,COM,commune,...,290.0,748.0,46.009,5.428,46.009,5.428,Rural à habitat dispersé,"Abergementais, Abergementaises",https://fr.wikipedia.org/wiki/fr:L'Abergement-...,https://villedereve.fr/ville/01002-l-abergemen...
2,2,01004,Ambérieu-en-Bugey,Ambérieu-en-Bugey,à Ambérieu-en-Bugey,d'Ambérieu-en-Bugey,amberieu-en-bugey,AMBÉRIEU-EN-BUGEY,COM,commune,...,237.0,753.0,45.961,5.373,45.961,5.373,Centres urbains intermédiaires,"Ambarrois, Ambarroises",https://fr.wikipedia.org/wiki/fr:Ambérieu-en-B...,https://villedereve.fr/ville/01004-amberieu-en...
3,3,01005,Ambérieux-en-Dombes,Ambérieux-en-Dombes,à Ambérieux-en-Dombes,d'Ambérieux-en-Dombes,amberieux-en-dombes,AMBÉRIEUX-EN-DOMBES,COM,commune,...,265.0,302.0,45.996,4.912,45.996,4.912,Bourgs ruraux,Ambarrois,https://fr.wikipedia.org/wiki/fr:Ambérieux-en-...,https://villedereve.fr/ville/01005-amberieux-e...
4,4,01006,Ambléon,Ambléon,à Ambléon,d'Ambléon,ambleon,AMBLÉON,COM,commune,...,330.0,940.0,45.750,5.594,45.750,5.594,Rural à habitat dispersé,Ambléonais,https://fr.wikipedia.org/wiki/fr:Ambléon,https://villedereve.fr/ville/01006-ambleon


## Nettoyage

In [12]:
incendies[COL_INSEE_INCENDIES] = incendies[COL_INSEE_INCENDIES].astype(str).str.zfill(5)
communes[COL_INSEE_COMMUNES] = communes[COL_INSEE_COMMUNES].astype(str).str.zfill(5)

incendies[COL_DATE] = pd.to_datetime(incendies[COL_DATE], errors="coerce", dayfirst=True)
incendies[COL_SURFACE] = pd.to_numeric(incendies[COL_SURFACE], errors="coerce")
incendies["annee"] = incendies[COL_DATE].dt.year

incendies = incendies.drop_duplicates()

incendies.shape


(140248, 25)

## Création de la base SQLite

In [13]:
conn = sqlite3.connect("incendies.db")

communes.to_sql("communes", conn, if_exists="replace", index=False)
incendies.to_sql("incendies", conn, if_exists="replace", index=False)


140248

## Vérification

In [14]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)


,name
0,communes
1,incendies


In [15]:
pd.read_sql('SELECT * FROM incendies LIMIT 5', conn)


,Année,Numéro,Département,Code INSEE,Nom de la commune,Date de première alerte,Surface parcourue (m2),Surface forêt (m2),Surface maquis garrigues (m2),Autres surfaces naturelles hors forêt (m2),...,Surfaces non boisées (m2),Précision des surfaces,Type de peuplement,Nature,Décès ou bâtiments touchés,Nombre de décès,Nombre de bâtiments totalement détruits,Nombre de bâtiments partiellement détruits,Précision de la donnée,annee
0,1973,8,06,06060,Falicon,1973-09-01 13:50:00,10000,4700.0,4800.0,0.0,...,500.0,None,1.0,NaN,None,None,None,None,None,1973.0
1,1973,1871,83,83038,Châteaudouble,NaN,30000,14100.0,14400.0,0.0,...,1500.0,None,3.0,NaN,None,None,None,None,None,NaN
2,1973,1872,83,83078,Moissac-Bellevue,NaN,6000,2820.0,2880.0,0.0,...,300.0,None,1.0,Involontaire (travaux),None,None,None,None,None,NaN
3,1973,1639,34,34163,Montarnaud,NaN,100000,47000.0,48000.0,0.0,...,5000.0,None,1.0,NaN,None,None,None,None,None,NaN
4,1973,9,06,06012,Beausoleil,NaN,40000,18800.0,19200.0,0.0,...,2000.0,None,5.0,NaN,None,None,None,None,None,NaN


In [17]:
query = f'''
SELECT i.annee, COUNT(*) as nb_incendies, SUM(i."{COL_SURFACE}") as surface_totale
FROM incendies i
JOIN communes c ON i."{COL_INSEE_INCENDIES}" = c."{COL_INSEE_COMMUNES}"
GROUP BY i.annee
ORDER BY i.annee DESC
LIMIT 10
'''
pd.read_sql(query, conn)


,annee,nb_incendies,surface_totale
0,2024.0,507,15451503
1,2023.0,1123,21476342
2,2022.0,1788,390228375
3,2021.0,803,18457802
4,2020.0,1249,66449831
5,2019.0,1021,42315944
6,2018.0,763,29349279
7,2017.0,1076,61115149
8,2016.0,1123,84160073
9,2015.0,1279,27530255


In [ ]:
conn.close()
